# **Air University Multan Campus**<br>

**LAB 7 Task<br>**

**Subject Name & Code:** SE425 Data Mining<br>
**Department:** Computer Science<br>
**Program:** BS Software Engineering<br>
**Semester:** 7 Fall 2026<br>
**Instructor:** Fazle Hanan<br>
**Date & Week:** 17 October 2026 & Week 7<br>
**Lab:** B216<br>

### Student Details

Save this notebook as `RegID_Name_Lab7Task.ipynb` before you submit it.

**Registration ID:** 233670

**Name:** Sidra Ghulam Hussain


Complete the 6 tasks below. Together they form one ETL pipeline (Extract, Transform, Load) that ends with a clean, merged data file. Write your code under each `# TODO` line and run the cell. No expected output is given — read each task carefully and check your own results. Run the tasks in order: later tasks use the variables created in earlier tasks. Do not load any file other than the three provided files. Marking criteria are in the Lab 7 Task Handout rubric.

### About the data

Real data from the `nycflights13` package, released under CC0 (https://cran.r-project.org/web/packages/nycflights13/).

* `nyc_flights_sample.csv` — 936 flights that left JFK airport on 1, 2 and 3 January 2013 (source: Bureau of Transportation Statistics). Columns: `month`, `day`, `carrier` (two-letter airline code), `flight` (flight number), `tailnum` (plane tail number), `dest` (destination airport), `dep_delay` and `arr_delay` (minutes; a negative value means early), `air_time` (minutes in the air) and `distance` (miles).
* `nyc_planes.csv` — 3322 planes (source: FAA Aircraft Registry). Columns: `tailnum`, `year` (year built), `type`, `manufacturer`, `model`, `engines`, `seats`, `speed` (average cruising speed in mph) and `engine` (engine type).
* `nyc_airlines.csv` — 16 airlines. Columns: `carrier` and `name`.

Changes made for this lab: only the flights from JFK on the first three days of January were kept, and only the columns listed above; nothing else was altered. A blank value in the CSV files is a missing value. The files do not record why a value is missing; filling is used here to practise the methods.

In [3]:
# Setup cell — run this first. Do not modify.
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# In Google Colab: when the upload window opens, choose the three CSV files
# (nyc_flights_sample.csv, nyc_planes.csv, nyc_airlines.csv).
# On your own computer (Jupyter / Anaconda): keep the three files in the notebook's folder; the upload step is skipped.
try:
    from google.colab import files
    files.upload()
except ImportError:
    pass

flights_df = pd.read_csv("nyc_flights_sample.csv")
planes_df = pd.read_csv("nyc_planes.csv")
airlines_df = pd.read_csv("nyc_airlines.csv")
print(flights_df.shape, planes_df.shape, airlines_df.shape)
flights_df.head()

Saving nyc_airlines.csv to nyc_airlines.csv
Saving nyc_flights_sample.csv to nyc_flights_sample.csv
Saving nyc_planes.csv to nyc_planes.csv
(936, 10) (3322, 9) (16, 2)


,month,day,carrier,flight,tailnum,dest,dep_delay,arr_delay,air_time,distance
0,1,1,AA,1141,N619AA,MIA,2.0,33.0,160.0,1089
1,1,1,B6,725,N804JB,BQN,-1.0,-18.0,183.0,1576
2,1,1,B6,79,N593JB,MCO,-3.0,-8.0,140.0,944
3,1,1,B6,49,N793JB,PBI,-2.0,-2.0,149.0,1028
4,1,1,B6,71,N657JB,TPA,-2.0,-3.0,158.0,1005


### Task 7.1 — Extract and inspect (10 marks)

Using `planes_df` and `flights_df`:
1. Store the percentage of missing values of every column of `planes_df`, rounded to 1 decimal place, as a pandas Series called `planes_missing_pct` and print it.
2. Drop from `planes_df` every column in which more than 50% of the values are missing (find the columns with code; do not type their names) and print the names of the dropped columns.
3. Print the number of missing values of every column of `flights_df` that has at least one missing value.
4. A flight with no `tailnum` cannot be linked to a plane. Print how many such flights there are, remove them from `flights_df`, and print the number of rows that remain.

In [4]:
# TODO: measure the missing values, drop the mostly-empty columns of planes_df, and remove the flights that have no tailnum

# isna().mean() is the share of missing cells; x 100 gives a percentage
planes_missing_pct = (planes_df.isna().mean() * 100).round(1)
print("Missing values in planes_df (%):")
print(planes_missing_pct)

# Find the columns with code instead of typing their names
mostly_empty_columns = planes_df.columns[planes_df.isna().mean() > 0.5]
planes_df = planes_df.drop(columns=mostly_empty_columns)
print("\nDropped columns:", mostly_empty_columns.tolist())

flights_missing = flights_df.isna().sum()
print("\nMissing values in flights_df:")
print(flights_missing[flights_missing > 0])

print("\nFlights with no tailnum:", flights_df["tailnum"].isna().sum())
flights_df = flights_df.dropna(subset=["tailnum"])
print("Rows that remain:", len(flights_df))

Missing values in planes_df (%):
tailnum          0.0
year             2.1
type             0.0
manufacturer     0.0
model            0.0
engines          0.0
seats            0.0
speed           99.3
engine           0.0
dtype: float64

Dropped columns: ['speed']

Missing values in flights_df:
tailnum      1
dep_delay    2
arr_delay    7
air_time     7
dtype: int64

Flights with no tailnum: 1
Rows that remain: 935


### Task 7.2 — Transform: fill the missing values (20 marks)

Create `flights_clean` as a copy of `flights_df`. In `flights_clean`:
1. Select (and keep) the rows whose `air_time` is missing before you fill anything, for use in step 4.
2. Replace the missing values of `dep_delay` and `arr_delay` with the median of the known values of each column (compute each median before filling).
3. Replace the missing values of `air_time` with the mean of the known `air_time` values of the flights to the same `dest`, rounded to 3 decimal places.
4. Print the number of missing values left in `dep_delay`, `arr_delay` and `air_time`, and for the flights whose `air_time` was missing print `dest` and the filled `air_time`.

In [5]:
# TODO: copy flights_df, fill dep_delay and arr_delay with the median, fill air_time with the mean of the same dest, and report the results

# Work on a copy so flights_df stays unchanged
flights_clean = flights_df.copy()

# The index labels let us find these rows again in step 4
air_time_missing_index = flights_clean.index[flights_clean["air_time"].isna()]

# Compute each median before filling (pandas skips NaN)
dep_delay_median = flights_clean["dep_delay"].median()
arr_delay_median = flights_clean["arr_delay"].median()
flights_clean["dep_delay"] = flights_clean["dep_delay"].fillna(dep_delay_median)
flights_clean["arr_delay"] = flights_clean["arr_delay"].fillna(arr_delay_median)

# groupby + transform("mean") gives every row the mean air_time of its own dest
dest_air_time_mean = flights_clean.groupby("dest")["air_time"].transform("mean").round(3)
flights_clean["air_time"] = flights_clean["air_time"].fillna(dest_air_time_mean)

print("Missing dep_delay:", flights_clean["dep_delay"].isna().sum())
print("Missing arr_delay:", flights_clean["arr_delay"].isna().sum())
print("Missing air_time: ", flights_clean["air_time"].isna().sum())

print("\nFlights whose air_time was missing (dest and filled air_time):")
print(flights_clean.loc[air_time_missing_index, ["dest", "air_time"]])

Missing dep_delay: 0
Missing arr_delay: 0
Missing air_time:  0

Flights whose air_time was missing (dest and filled air_time):
    dest  air_time
243  DFW   233.750
296  FLL   162.689
546  DFW   233.750
565  SAT   251.000
593  RSW   173.235
867  SAT   251.000


### Task 7.3 — Transform: binning (20 marks)

Using `flights_clean`:
1. Create the column `distance_group` by dividing `distance` into 3 equal-width bins with `pd.cut`, labelled `short`, `medium` and `long`, and print the number of flights in each group in that order.
2. Smooth `air_time` by bin means: sort the rows by `air_time` (use `kind="stable"`), divide the sorted rows into 5 equal-frequency bins with `np.array_split`, and store the mean of each bin, rounded to 3 decimal places, in a new column `air_time_smooth` for every row of that bin. Hint: split the index labels of the sorted rows.
3. Print the number of rows and the mean of each of the 5 bins, and print the number of distinct values in `air_time` and in `air_time_smooth`.

In [6]:
# TODO: build distance_group with pd.cut, build air_time_smooth with 5 equal-frequency bins, and print the bin sizes, the bin means and the distinct-value counts

# pd.cut splits the range (max - min) into intervals of the same width
distance_labels = ["short", "medium", "long"]
flights_clean["distance_group"] = pd.cut(flights_clean["distance"], bins=3, labels=distance_labels)

# sort_index() keeps the category order short, medium, long
print("Flights per distance_group:")
print(flights_clean["distance_group"].value_counts().sort_index())

# Sort the rows by air_time; kind="stable" keeps tied rows in their original order
sorted_index = flights_clean.sort_values("air_time", kind="stable").index

# Split the index labels of the sorted rows into 5 bins with the same number of rows (+-1)
air_time_bins = np.array_split(sorted_index, 5)

# Every row of a bin gets the bin mean, rounded to 3 decimals
flights_clean["air_time_smooth"] = np.nan
bin_sizes = []
bin_means = []
for bin_labels in air_time_bins:
    bin_mean = round(flights_clean.loc[bin_labels, "air_time"].mean(), 3)
    flights_clean.loc[bin_labels, "air_time_smooth"] = bin_mean
    bin_sizes.append(len(bin_labels))
    bin_means.append(bin_mean)

print("\nRows and mean of each air_time bin:")
for i, (size, mean) in enumerate(zip(bin_sizes, bin_means), start=1):
    print(f"Bin {i}: rows = {size}, mean = {mean}")

print("\nDistinct values in air_time:       ", flights_clean["air_time"].nunique())
print("Distinct values in air_time_smooth:", flights_clean["air_time_smooth"].nunique())

Flights per distance_group:
distance_group
short     649
medium    283
long        3
Name: count, dtype: int64

Rows and mean of each air_time bin:
Bin 1: rows = 187, mean = 48.374
Bin 2: rows = 187, mean = 111.455
Bin 3: rows = 187, mean = 164.197
Bin 4: rows = 187, mean = 259.158
Bin 5: rows = 187, mean = 346.738

Distinct values in air_time:        291
Distinct values in air_time_smooth: 5


### Task 7.4 — Transform: normalization and standardization (15 marks)

Using `flights_clean`:
1. Create `distance_norm` and `air_time_norm` by min-max normalization of `distance` and `air_time`, directly from the formula.
2. Create `dep_delay_z` and `arr_delay_z` as the z-scores of `dep_delay` and `arr_delay`, directly from the formula (use the population standard deviation, `ddof=0`).
3. Confirm the results with scikit-learn: apply `MinMaxScaler` to the two columns of step 1 and `StandardScaler` to the two columns of step 2, and print the largest absolute difference for each scaler.
4. Print the minimum and the maximum of `distance_norm` and `air_time_norm`, and the mean and the population standard deviation of `dep_delay_z` and `arr_delay_z`, all rounded to 6 decimal places.

In [7]:
# TODO: build the two min-max columns and the two z-score columns by formula, compare them with scikit-learn, and print the checks

for column in ["distance", "air_time"]:
    col = flights_clean[column]
    flights_clean[column + "_norm"] = (col - col.min()) / (col.max() - col.min())

for column in ["dep_delay", "arr_delay"]:
    col = flights_clean[column]
    flights_clean[column + "_z"] = (col - col.mean()) / col.std(ddof=0)

# Both scalers take a 2-D table, so the two columns are passed together
sk_min_max = MinMaxScaler().fit_transform(flights_clean[["distance", "air_time"]])
sk_z = StandardScaler().fit_transform(flights_clean[["dep_delay", "arr_delay"]])

# Largest absolute difference between the manual and the scikit-learn results (should be zero or tiny)
print("Largest |manual - sklearn| min-max:",
      np.abs(flights_clean[["distance_norm", "air_time_norm"]].to_numpy() - sk_min_max).max())
print("Largest |manual - sklearn| z-score:",
      np.abs(flights_clean[["dep_delay_z", "arr_delay_z"]].to_numpy() - sk_z).max())

# min-max columns must run from 0 to 1
for column in ["distance_norm", "air_time_norm"]:
    print(f"{column}: min = {flights_clean[column].min():.6f}, max = {flights_clean[column].max():.6f}")

# z-score columns must have mean 0 and population std 1
for column in ["dep_delay_z", "arr_delay_z"]:
    print(f"{column}: mean = {flights_clean[column].mean():.6f}, std = {flights_clean[column].std(ddof=0):.6f}")

Largest |manual - sklearn| min-max: 1.1102230246251565e-16
Largest |manual - sklearn| z-score: 1.4210854715202004e-14
distance_norm: min = 0.000000, max = 1.000000
air_time_norm: min = 0.000000, max = 1.000000
dep_delay_z: mean = -0.000000, std = 1.000000
arr_delay_z: mean = 0.000000, std = 1.000000


### Task 7.5 — Transform: merge the tables and complete the result (25 marks)

1. Create `planes_small` from `planes_df`: rename the column `year` to `plane_year` and keep only `tailnum`, `plane_year`, `manufacturer`, `model`, `seats` and `engine`. Create `airlines_small` from `airlines_df` by renaming the column `name` to `airline_name`.
2. Print whether `tailnum` is unique in `planes_small` and whether `carrier` is unique in `airlines_small`.
3. Merge `flights_clean` with `planes_small` on `tailnum` with a left join, `validate="many_to_one"` and `indicator=True`. Print the number of rows of each kind in the `_merge` column, then remove that column and store the result as `with_planes`.
4. Merge `with_planes` with `airlines_small` on `carrier` with a left join and `validate="many_to_one"`, and store the result as `merged_df`. Print its shape and whether it has the same number of rows as `flights_clean`.
5. Print the number of missing values in each of `plane_year`, `seats`, `manufacturer`, `model` and `engine` of `merged_df`. Then fill `plane_year` and `seats` with the median of their known values in `merged_df` (compute each median before filling) and fill `manufacturer`, `model` and `engine` with the text `Unknown`. Print the total number of missing values in `merged_df`.

In [8]:
# TODO: prepare planes_small and airlines_small, check the keys, merge into with_planes and merged_df, then complete the plane columns

# Rename year so it cannot be confused with a flight year, then keep only the needed columns
planes_small = planes_df.rename(columns={"year": "plane_year"})[
    ["tailnum", "plane_year", "manufacturer", "model", "seats", "engine"]
]
# Rename name so the column says what it holds
airlines_small = airlines_df.rename(columns={"name": "airline_name"})

print("tailnum unique in planes_small:  ", planes_small["tailnum"].is_unique)
print("carrier unique in airlines_small:", airlines_small["carrier"].is_unique)

# many_to_one: many flights share one plane; indicator=True adds the _merge column
with_planes = flights_clean.merge(
    planes_small, on="tailnum", how="left", validate="many_to_one", indicator=True
)
print("\nRows of each kind in _merge:")
print(with_planes["_merge"].value_counts())

# The _merge column was only needed for the check
with_planes = with_planes.drop(columns="_merge")

merged_df = with_planes.merge(airlines_small, on="carrier", how="left", validate="many_to_one")
print("\nMerged table shape:", merged_df.shape)
print("Same number of rows as flights_clean:", len(merged_df) == len(flights_clean))

plane_columns = ["plane_year", "seats", "manufacturer", "model", "engine"]
print("\nMissing values before filling:")
print(merged_df[plane_columns].isna().sum())

# Numeric columns: median of the known values (computed before filling)
plane_year_median = merged_df["plane_year"].median()
seats_median = merged_df["seats"].median()
merged_df["plane_year"] = merged_df["plane_year"].fillna(plane_year_median)
merged_df["seats"] = merged_df["seats"].fillna(seats_median)

# Text columns: the global constant "Unknown"
for column in ["manufacturer", "model", "engine"]:
    merged_df[column] = merged_df[column].fillna("Unknown")

print("\nTotal missing values in merged_df:", merged_df.isna().sum().sum())

tailnum unique in planes_small:   True
carrier unique in airlines_small: True

Rows of each kind in _merge:
_merge
both          791
left_only     144
right_only      0
Name: count, dtype: int64

Merged table shape: (935, 22)
Same number of rows as flights_clean: True

Missing values before filling:
plane_year      155
seats           144
manufacturer    144
model           144
engine          144
dtype: int64

Total missing values in merged_df: 0


### Task 7.6 — Load: save and check the clean dataset (10 marks)

1. Save `merged_df` as `clean_flights_merged.csv` without the index column.
2. Read the saved file back as `check_df` and print its shape, its column names as a list, the total number of missing values and the number of duplicated rows.
3. Print whether `check_df` has the same number of rows and columns as `merged_df`.

In Google Colab the saved file stays in the session storage (the Files panel on the left) and is deleted when the session ends.

In [9]:
# TODO: save merged_df to clean_flights_merged.csv, read it back as check_df, and print the checks

# without the index column
merged_df.to_csv("clean_flights_merged.csv", index=False)

# read the file back and check it
check_df = pd.read_csv("clean_flights_merged.csv")
print("Shape:", check_df.shape)
print("Columns:", check_df.columns.tolist())
print("Total missing values:", check_df.isna().sum().sum())
print("Duplicated rows:", check_df.duplicated().sum())

# with merged_df
print("Same number of rows and columns as merged_df:", check_df.shape == merged_df.shape)

Shape: (935, 22)
Columns: ['month', 'day', 'carrier', 'flight', 'tailnum', 'dest', 'dep_delay', 'arr_delay', 'air_time', 'distance', 'distance_group', 'air_time_smooth', 'distance_norm', 'air_time_norm', 'dep_delay_z', 'arr_delay_z', 'plane_year', 'manufacturer', 'model', 'seats', 'engine', 'airline_name']
Total missing values: 0
Duplicated rows: 0
Same number of rows and columns as merged_df: True
